# RQ3: Feature Importance for Attrition Risk Prediction

## Research Question
**Which features have the strongest predictive power for attrition risk?**

## Hypothesis
Job satisfaction and years since last promotion are top predictors of attrition.

## Objective
Build predictive models and extract feature importance scores to identify the strongest drivers of employee attrition.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# Load and preprocess data
df = pd.read_csv('Employee_Attrition.csv')

# Encode target
df['Attrition_Binary'] = (df['Attrition'] == 'Yes').astype(int)

# Encode categorical variables
le_dict = {}
categorical_cols = df.select_dtypes(include=['object']).columns
df_encoded = df.copy()

for col in categorical_cols:
    if col != 'Attrition':
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df[col])
        le_dict[col] = le

# Prepare features and target
numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in numeric_cols if col not in ['Attrition_Binary']]

X = df_encoded[feature_cols]
y = df_encoded['Attrition_Binary']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset prepared: {X.shape[0]} rows × {X.shape[1]} features")

## 1. Random Forest Feature Importance

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

# Extract feature importance
rf_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Random Forest Feature Importance (Top 15):")
print(rf_importance.head(15))

# Visualization
fig, ax = plt.subplots(figsize=(12, 8))
rf_importance.head(15).plot(x='Feature', y='Importance', kind='barh', ax=ax, color='steelblue', legend=False)
ax.set_title('Top 15 Features - Random Forest Feature Importance', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Model performance
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
y_pred_rf = rf_model.predict(X_test)
print(f"\nRandom Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Random Forest ROC-AUC: {roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1]):.4f}")

## 2. Logistic Regression Coefficients

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

# Extract coefficients (absolute values for importance)
lr_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr_model.coef_[0],
    'Abs_Coefficient': np.abs(lr_model.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

print("Logistic Regression Feature Importance (Top 15):")
print(lr_importance.head(15)[['Feature', 'Coefficient', 'Abs_Coefficient']])

# Visualization
fig, ax = plt.subplots(figsize=(12, 8))
lr_importance.head(15).plot(x='Feature', y='Coefficient', kind='barh', ax=ax, color='coral', legend=False)
ax.set_title('Top 15 Features - Logistic Regression Coefficients', fontsize=14, fontweight='bold')
ax.set_xlabel('Coefficient Value')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Model performance
y_pred_lr = lr_model.predict(X_test_scaled)
print(f"\nLogistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Logistic Regression ROC-AUC: {roc_auc_score(y_test, lr_model.predict_proba(X_test_scaled)[:, 1]):.4f}")

# Combine importances (normalized)
rf_norm = rf_importance.set_index('Feature')['Importance'] / rf_importance['Importance'].sum()
lr_norm = lr_importance.set_index('Feature')['Abs_Coefficient'] / lr_importance['Abs_Coefficient'].sum()

# Compare top features from both models
comparison = pd.DataFrame({
    'Random_Forest': rf_importance.set_index('Feature')['Importance'],
    'Logistic_Regression': lr_importance.set_index('Feature')['Abs_Coefficient']
}).fillna(0)

# Normalize for comparison
comparison_norm = comparison / comparison.max()
comparison_norm = comparison_norm.sort_values('Random_Forest', ascending=False).head(15)

# Visualization
fig, ax = plt.subplots(figsize=(12, 8))
comparison_norm.plot(kind='barh', ax=ax, color=['steelblue', 'coral'])
ax.set_title('Feature Importance Comparison: Random Forest vs Logistic Regression (Top 15)', fontsize=14, fontweight='bold')
ax.set_xlabel('Normalized Importance')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

print("\nTop Features Agreement Between Models:")
print(comparison_norm)

print("\n" + "="*80)
print("RQ3: RESEARCH QUESTION 3 - KEY FINDINGS")
print("="*80)

print("\nTOP 10 FEATURES PREDICTING ATTRITION (Random Forest):")
for idx, row in rf_importance.head(10).iterrows():
    print(f"  {idx+1}. {row['Feature']}: {row['Importance']:.4f}")

print("\nKEY INSIGHTS:")
print(f"  - Cumulative importance of top 5 features: {rf_importance.head(5)['Importance'].sum():.4f}")
print(f"  - Cumulative importance of top 10 features: {rf_importance.head(10)['Importance'].sum():.4f}")

# Check hypothesis
job_sat_rank = rf_importance[rf_importance['Feature'] == 'JobSatisfaction'].index[0] + 1 if 'JobSatisfaction' in rf_importance['Feature'].values else None
promotion_rank = rf_importance[rf_importance['Feature'] == 'YearsSinceLastPromotion'].index[0] + 1 if 'YearsSinceLastPromotion' in rf_importance['Feature'].values else None

print("\n" + "="*80)
print("HYPOTHESIS VALIDATION:")
print("="*80)
if job_sat_rank and job_sat_rank <= 5:
    print(f"✓ SUPPORTED: Job Satisfaction is a top predictor (Rank #{job_sat_rank})")
elif job_sat_rank:
    print(f"✗ PARTIALLY SUPPORTED: Job Satisfaction is important but ranked #{job_sat_rank}")

if promotion_rank and promotion_rank <= 5:
    print(f"✓ SUPPORTED: Years Since Last Promotion is a top predictor (Rank #{promotion_rank})")
elif promotion_rank:
    print(f"✗ PARTIALLY SUPPORTED: Years Since Last Promotion ranked #{promotion_rank}")

print("\n✓ Multiple strong predictors identified beyond job satisfaction and promotion")